# Torsion Scan Guardian — Colab runbook

Zero-cost end-to-end execution of the Phase 0–5 pipeline on Google Colab's free T4 GPU.

**Before running:** `Runtime → Change runtime type → T4 GPU`.

**Validated** on 2026-05-17: Phase 5 demo completes in 466 s (~2× faster than CPU) with all 11 unit tests passing. See REPORT.md §13.7 for the cross-environment comparison.

Each cell is independent; you can re-run from any point after the install + xtb cells finish.

## Colab-specific gotchas baked into this notebook
- `%%bash` cells include an explicit `cd` because `%%bash` spawns a fresh subprocess that doesn't inherit IPython's `%cd`.
- Python scripts that touch matplotlib are invoked with `MPLBACKEND=Agg` to avoid the missing-display error.
- `numpy` is pinned in the pip install for predictable resolution against `mace-torch`.
- `condacolab.install()` **restarts the kernel** — re-run cells 1–3 after it finishes (cell 4).

## 1. Verify GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Torch version:', torch.__version__)

## 2. Get the project source

Two options:
- **A.** Mount Google Drive and clone the repo there (preferred — outputs persist across sessions).
- **B.** Clone directly into Colab's ephemeral storage (faster, but you lose results when the session ends).

The Drive path is hardcoded to `/content/drive/MyDrive/torsion-scan-guardian` and is reused by the `%%bash` cells below. If you change the location, update `REPO_DIR` accordingly.

In [ ]:
# --- Option A: persistent (recommended) ---
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive
REPO_URL = 'https://github.com/Sathapana/torsion_scan_guardian.git'
REPO_DIR = '/content/drive/MyDrive/torsion-scan-guardian'
!test -d torsion-scan-guardian || git clone $REPO_URL torsion-scan-guardian
%cd torsion-scan-guardian

In [ ]:
# Sanity check: clone landed and is at the right path.
import os
if os.path.exists(REPO_DIR) and os.path.exists(os.path.join(REPO_DIR, 'pyproject.toml')):
    print(f'OK — repository present at {REPO_DIR}')
else:
    raise RuntimeError(f'Repository NOT at {REPO_DIR}; check REPO_URL and your git access to it.')

In [ ]:
# --- Option B: ephemeral ---
# %cd /content
# REPO_URL = 'https://github.com/Sathapana/torsion_scan_guardian.git'
# REPO_DIR = '/content/torsion_scan_guardian'
# !git clone $REPO_URL
# %cd torsion_scan_guardian

## 3. Install Python dependencies

Colab already ships PyTorch with CUDA. We pip-install everything else.

**~3–4 minutes the first time, then cached for subsequent restarts in the same session.**

In [ ]:
%%bash
pip install -q --upgrade pip
pip install -q \
    mace-torch \
    ase \
    rdkit \
    matplotlib \
    pandas \
    pydantic \
    pyyaml \
    tqdm \
    pytest \
    numpy
cd /content/drive/MyDrive/torsion-scan-guardian
pip install -q -e .

## 4. Install xtb (for the GFN-FF oracle)

Colab doesn't have `xtb` by default. We install via `conda-forge` using `condacolab`.

**`condacolab.install()` restarts the kernel — that's expected. Re-run cells 1–3 (verify GPU, mount Drive, sanity-check clone, pip install) before continuing to cell 4b.**

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()  # restarts the kernel — re-run cells 1–3 after this

In [ ]:
# 4b — after kernel restart, install xtb itself.
!conda install -y -c conda-forge xtb 2>&1 | tail -5
!which xtb && xtb --version 2>&1 | head -5

## 5. Pre-download MACE-OFF small

One-shot ~7 MB fetch; cached at `~/.cache/mace/`. On Colab this loads as float64 by default (more accurate, slightly different ensemble disagreement than the float32 local CPU runs — see REPORT §13.7).

In [ ]:
import torch
from mace.calculators import mace_off
calc = mace_off(model='medium', device='cuda' if torch.cuda.is_available() else 'cpu')
print('MACE-OFF small loaded on', calc.device if hasattr(calc, 'device') else 'unknown')

## 6. Run unit tests (smoke check)

In [ ]:
!python -m pytest -q tests/ -k 'not test_ensemble_predict_h2o'

## 7. Phase 5 demo — sulfanilamide AL run with cloud + safeguard

Same configuration as REPORT §12.5, now on GPU. Validated wall time: **~470 s** (vs 974 s on CPU).

Cells 7a–7d each `cd` into `$REPO_DIR` explicitly — `%%bash` cells do NOT inherit `%cd` from earlier cells.

In [ ]:
%%bash
# Step 7a: build the seed dataset (~3 min).
cd /content/drive/MyDrive/torsion-scan-guardian
python scripts/build_seed_dataset.py \
    --smiles 'Nc1ccc(S(=O)(=O)N)cc1' \
    --out data/seed/sulfanilamide_seed.xyz

In [ ]:
%%bash
# Step 7b: fine-tune 3 ensemble members on GPU (~1–2 min total).
cd /content/drive/MyDrive/torsion-scan-guardian
for SEED in 0 1 2 3 4; do
  MPLBACKEND=Agg python scripts/finetune_member.py \
      --seed $SEED --epochs 5 --lr 5e-4 --foundation-size medium \
      --train-file data/seed/sulfanilamide_seed.xyz \
      --out-root runs/finetune_sulf \
      --device cuda
done
ls runs/finetune_sulf_medium/*/checkpoints/*.model

In [ ]:
%%bash
# Step 7c: run the 5-cycle AL loop with cloud acquisitions + safeguarded fine-tune.
cd /content/drive/MyDrive/torsion-scan-guardian
MPLBACKEND=Agg python -m guardian.cli \
    --config config/default.yaml \
    --smiles 'Nc1ccc(S(=O)(=O)N)cc1' \
    --steps 4000 --temperature 300 --threshold 0.05 \
    --max-triggers 5 --cooldown-steps 200 \
    --cloud-size 5 --cloud-jitter 0.05 \
    --ft-regression-tol 0.10 \
    --checkpoints runs/finetune_sulf_medium/member_seed0/checkpoints/*.model \
                  runs/finetune_sulf_medium/member_seed1/checkpoints/*.model \
                  runs/finetune_sulf_medium/member_seed2/checkpoints/*.model \
    --online-finetune \
    --seed-data-file data/seed/sulfanilamide_seed.xyz \
    --finetune-epochs 2 --finetune-lr 1e-4 \
    --run-dir runs/sulf_phase5_colab

In [ ]:
# Step 7d: analysis figures (uses matplotlib — needs Agg backend on headless Colab).
!MPLBACKEND=Agg python scripts/analyse_al_demo.py runs/sulf_phase5_colab

In [ ]:
# Display the timeline figure inline.
from IPython.display import Image
Image('figures/sulf_phase5_colab_timeline.png')

## 8. (Optional) Phase 6 first attempt — glycine zwitterion

Same machinery, a tougher molecule. The §12.7 candidate. **Note:** charged species like glycine zwitterion may need a `--chrg` flag added to the xtb wrapper for GFN2-xTB or DFT oracles. For GFN-FF (what this notebook uses), the net-zero charge is autodetected and no code change is needed.

In [ ]:
# Smoke check: does the SMILES parse and embed?
from rdkit import Chem
from rdkit.Chem import AllChem
mol = Chem.MolFromSmiles('[NH3+]CC(=O)[O-]')
mol = Chem.AddHs(mol)
AllChem.EmbedMolecule(mol, randomSeed=0)
print('Atoms:', mol.GetNumAtoms(), '  formal charge sum:', sum(a.GetFormalCharge() for a in mol.GetAtoms()))

## 9. Bundle results for download

Zips up the small, version-worthy artifacts (figures, summary JSONs, optional new seed datasets) and triggers a browser download. Unzip on your local machine into the repo, then `git add` + `git commit` + `git push` from there — that avoids putting GitHub credentials inside Colab.

In [ ]:
import shutil, os
from google.colab import files

bundle_dir = '/content/guardian_results'
os.makedirs(bundle_dir, exist_ok=True)

# Copy only what's worth versioning.
!cp -v figures/*.png $bundle_dir/ 2>/dev/null || true
!cp -v runs/*/summary.json $bundle_dir/ 2>/dev/null || true
!cp -v data/seed/*.xyz $bundle_dir/ 2>/dev/null || true

shutil.make_archive('/content/guardian_results', 'zip', bundle_dir)
files.download('/content/guardian_results.zip')